# Comprehensive Campus IoT Energy Management Analysis

This notebook contains the complete workflow for the Campus IoT Energy Management project. It consolidates the multi-file Python project into a single, executable notebook. The process includes:

1.  **Data Preprocessing**: Loading, cleaning, and feature engineering on the campus energy dataset.
2.  **Model Definition**: Defining the LSTM, simple machine learning models, and the ensemble that combines them.
3.  **Model Training**: Training the LSTM and other models on the preprocessed data.
4.  **Evaluation & Analysis**: Evaluating the ensemble and individual models, analyzing performance, and generating insights.

## 1. Imports and Setup

First, we import all the necessary libraries and set up the environment. This includes libraries for data manipulation, machine learning, and plotting.

In [4]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings('ignore')
print(f"TensorFlow Version: {tf.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Scikit-learn Version: {sklearn.__version__}")

TensorFlow Version: 2.20.0
Pandas Version: 2.3.2
Scikit-learn Version: 1.7.2


## 2. Class Definitions

Here, we define all the classes from the project files. This encapsulates the logic for data preprocessing, model building, and ensembling.

### 2.1 `CampusEnergyPreprocessor`

This class handles all data loading, cleaning, feature engineering, and preparation steps for the campus energy dataset.

In [5]:
class CampusEnergyPreprocessor:
    def __init__(self, dataset_path):
        self.dataset_path = dataset_path
        self.scalers = {}
        self.encoders = {}
        
    def load_all_buildings_data(self):
        """Load the main all_buildings_power.csv file"""
        filepath = os.path.join(self.dataset_path, 'all_buildings_power.csv')
        if os.path.exists(filepath):
            data = pd.read_csv(filepath)
            print(f"Loaded all_buildings_power.csv: {data.shape}")
            return data
        else:
            raise FileNotFoundError("all_buildings_power.csv not found!")
    
    def create_simplified_dataset(self):
        """Create simplified dataset focusing on power consumption only"""
        # Load main power consumption data
        power_data = self.load_all_buildings_data()
        
        # Convert Unix timestamp to datetime
        power_data['timestamp'] = pd.to_datetime(power_data['timestamp'], unit='s')
        power_data = power_data.sort_values('timestamp').reset_index(drop=True)
        
        print(f"Date range: {power_data['timestamp'].min()} to {power_data['timestamp'].max()}")
        
        # Create unified dataset
        unified_data = []
        
        # Building mapping - focus on main buildings with good data
        building_columns = {
            'Academic': 'academic',
            'Boys_main': 'boys_main', 
            'Girls_main': 'girls_main',
            'Lecture': 'lecture',
            'Library': 'library'
        }
        
        for col, building_type in building_columns.items():
            if col in power_data.columns:
                # Extract power consumption for this building
                building_power = power_data[['timestamp', col]].copy()
                # Remove rows where power is NaN or 0
                building_power = building_power.dropna()
                building_power = building_power[building_power[col] > 0]
                
                if len(building_power) > 1000:  # Only include buildings with sufficient data
                    building_power['energy_consumption'] = building_power[col]
                    building_power['building_type'] = building_type
                    
                    # Add simple electrical parameters (estimated values)
                    building_power['voltage'] = 240.0 + np.random.normal(0, 5, len(building_power))
                    building_power['frequency'] = 50.0 + np.random.normal(0, 0.1, len(building_power))
                    building_power['power_factor'] = 0.95 + np.random.normal(0, 0.05, len(building_power))
                    
                    # Clip to realistic ranges
                    building_power['voltage'] = np.clip(building_power['voltage'], 220, 260)
                    building_power['frequency'] = np.clip(building_power['frequency'], 49, 51)
                    building_power['power_factor'] = np.clip(building_power['power_factor'], 0.8, 1.0)
                    
                    unified_data.append(building_power)
                    print(f"Added {building_type}: {len(building_power)} samples")
        
        # Combine all building data
        if unified_data:
            combined_data = pd.concat(unified_data, ignore_index=True)
            combined_data = combined_data.sort_values(['building_type', 'timestamp']).reset_index(drop=True)
            print(f"Combined dataset shape: {combined_data.shape}")
            return combined_data
        else:
            raise ValueError("No building data could be processed!")
    
    def add_temporal_features(self, data):
        """Add temporal features specific to campus energy patterns"""
        data['hour'] = data['timestamp'].dt.hour
        data['day'] = data['timestamp'].dt.day
        data['month'] = data['timestamp'].dt.month
        data['weekday'] = data['timestamp'].dt.weekday
        data['weekend'] = (data['weekday'] >= 5).astype(int)
        
        # Campus-specific features
        data['is_working_hour'] = ((data['hour'] >= 8) & (data['hour'] <= 18) & (data['weekday'] < 5)).astype(int)
        data['is_peak_hour'] = ((data['hour'] >= 9) & (data['hour'] <= 11) | 
                               (data['hour'] >= 14) & (data['hour'] <= 16)).astype(int)
        data['is_night'] = ((data['hour'] >= 22) | (data['hour'] <= 6)).astype(int)
        
        # Cyclical encoding
        data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
        data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
        data['month_sin'] = np.sin(2 * np.pi * data['month'] / 12)
        data['month_cos'] = np.cos(2 * np.pi * data['month'] / 12)
        
        return data
    
    def add_electrical_features(self, data):
        """Add derived electrical features"""
        # Power quality indicators
        data['voltage_deviation'] = abs(data['voltage'] - 240.0)
        data['frequency_deviation'] = abs(data['frequency'] - 50.0)
        data['low_power_factor'] = (data['power_factor'] < 0.9).astype(int)
        
        return data
    
    def add_lag_features_safe(self, data, target_col='energy_consumption'):
        """Add lag features safely with minimal data loss"""
        data = data.sort_values(['building_type', 'timestamp']).reset_index(drop=True)
        
        # Initialize lag columns
        lag_columns = [f'{target_col}_lag_1', f'{target_col}_lag_10', 
                      f'{target_col}_rolling_mean_10', f'{target_col}_rolling_std_10']
        for col in lag_columns:
            data[col] = np.nan
        
        for building in data['building_type'].unique():
            mask = data['building_type'] == building
            building_indices = data.index[mask]
            building_data = data.loc[mask, target_col]
            
            if len(building_data) > 20:  # Only process if sufficient data
                # Short-term lags (minimal data loss)
                data.loc[building_indices[1:], f'{target_col}_lag_1'] = building_data.iloc[:-1].values
                data.loc[building_indices[10:], f'{target_col}_lag_10'] = building_data.iloc[:-10].values
                
                # Rolling statistics
                rolling_mean = building_data.rolling(window=10, min_periods=5).mean()
                rolling_std = building_data.rolling(window=10, min_periods=5).std()
                
                data.loc[building_indices, f'{target_col}_rolling_mean_10'] = rolling_mean.values
                data.loc[building_indices, f'{target_col}_rolling_std_10'] = rolling_std.values
        
        return data
    
    def create_feature_subsets(self):
        """Define feature subsets for ensemble modeling"""
        temporal_features = [
            'hour', 'day', 'month', 'weekday', 'weekend',
            'is_working_hour', 'is_peak_hour', 'is_night',
            'hour_sin', 'hour_cos', 'month_sin', 'month_cos'
        ]
        
        electrical_features = [
            'voltage', 'frequency', 'power_factor',
            'voltage_deviation', 'frequency_deviation', 'low_power_factor'
        ]
        
        lag_features = [
            'energy_consumption_lag_1', 'energy_consumption_lag_10',
            'energy_consumption_rolling_mean_10', 'energy_consumption_rolling_std_10'
        ]
        
        building_features = ['building_type_encoded']
        
        all_features = temporal_features + electrical_features + lag_features + building_features
        
        return {
            'temporal': temporal_features,
            'electrical': electrical_features,
            'lag_features': lag_features,
            'building': building_features,
            'all_features': all_features
        }
    
    def prepare_complete_dataset(self, target_col='energy_consumption', sample_size=100000):
        """Complete preprocessing pipeline with sampling for manageable size"""
        print("Creating simplified dataset...")
        data = self.create_simplified_dataset()
        
        # Sample data to manageable size for training
        if len(data) > sample_size:
            # Sample proportionally from each building
            sampled_data = []
            for building in data['building_type'].unique():
                building_data = data[data['building_type'] == building]
                building_sample_size = min(len(building_data), sample_size // len(data['building_type'].unique()))
                building_sample = building_data.sample(n=building_sample_size, random_state=42)
                sampled_data.append(building_sample)
            data = pd.concat(sampled_data, ignore_index=True)
            data = data.sort_values(['building_type', 'timestamp']).reset_index(drop=True)
            print(f"Sampled data to: {data.shape}")
        
        print("Adding temporal features...")
        data = self.add_temporal_features(data)
        
        print("Adding electrical features...")
        data = self.add_electrical_features(data)
        
        print("Adding lag features safely...")
        data = self.add_lag_features_safe(data, target_col)
        
        # Encode building types
        le = LabelEncoder()
        data['building_type_encoded'] = le.fit_transform(data['building_type'])
        self.encoders['building_type'] = le
        
        # Handle missing values more conservatively
        print("Handling missing values...")
        initial_length = len(data)
        
        # Fill missing values with forward fill, then backward fill, then mean
        numeric_columns = data.select_dtypes(include=[np.number]).columns
        for col in numeric_columns:
            if col != target_col:  # Don't fill target column
                data[col] = data.groupby('building_type')[col].transform(
                    lambda x: x.fillna(method='ffill').fillna(method='bfill').fillna(x.mean())
                )
        
        # Only drop rows where target is missing
        data = data.dropna(subset=[target_col])
        
        # Drop rows with remaining NaN in critical features
        critical_features = ['energy_consumption_lag_1', target_col]
        data = data.dropna(subset=critical_features)
        
        print(f"Removed {initial_length - len(data)} rows with missing values")
        print(f"Final dataset shape: {data.shape}")
        
        if len(data) == 0:
            raise ValueError("All data was removed during preprocessing! Check your data quality.")
        
        return data
    
    def create_sequences(self, data, feature_cols, target_col, window_size=10):
        """Create sequences for LSTM with smaller window size"""
        # Reduce window size to prevent data loss
        all_X, all_y = [], []
        
        for building in data['building_type'].unique():
            building_data = data[data['building_type'] == building].copy()
            building_data = building_data.sort_values('timestamp').reset_index(drop=True)
            
            if len(building_data) < window_size + 1:
                print(f"Skipping {building}: insufficient data ({len(building_data)} samples)")
                continue
                
            features = building_data[feature_cols].values
            target = building_data[target_col].values
            
            # Check for any remaining NaN values
            if np.isnan(features).any() or np.isnan(target).any():
                print(f"Warning: NaN values found in {building} data")
                continue
            
            # Scale features for this building
            scaler = StandardScaler()
            features_scaled = scaler.fit_transform(features)
            self.scalers[f'features_{building}'] = scaler
            
            # Scale target
            target_scaler = StandardScaler()
            target_scaled = target_scaler.fit_transform(target.reshape(-1, 1)).flatten()
            self.scalers[f'target_{building}'] = target_scaler
            
            # Create sequences
            for i in range(window_size, len(features_scaled)):
                all_X.append(features_scaled[i-window_size:i])
                all_y.append(target_scaled[i])
            
            print(f"Created {len(features_scaled) - window_size} sequences for {building}")
        
        if len(all_X) == 0:
            raise ValueError("No sequences could be created! Check your data preprocessing.")
        
        return np.array(all_X), np.array(all_y)
    
    def split_data(self, X, y, test_size=0.2, val_size=0.1):
        """Split data maintaining temporal order"""
        total_samples = len(X)
        print(f"Total sequences available: {total_samples}")
        
        if total_samples == 0:
            raise ValueError("No data to split!")
        
        # Calculate split indices
        train_end = int(total_samples * (1 - test_size - val_size))
        val_end = int(total_samples * (1 - test_size))
        
        X_train = X[:train_end]
        X_val = X[train_end:val_end]
        X_test = X[val_end:]
        
        y_train = y[:train_end]
        y_val = y[train_end:val_end]
        y_test = y[val_end:]
        
        print(f"Data splits - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        
        return X_train, X_val, X_test, y_train, y_val, y_test

### 2.2 `DecodeLSTM`

This class defines the LSTM model architecture, based on the DECODE paper. It includes methods for building, training, and evaluating the model.

In [6]:
class DecodeLSTM:
    def __init__(self, input_shape):
        self.input_shape = input_shape
        self.model = None
        self.history = None
        
    def build_model(self):
        """Build LSTM model based on DECODE paper architecture"""
        model = Sequential([
            # First LSTM layer
            LSTM(64, return_sequences=True, input_shape=self.input_shape),
            BatchNormalization(),
            Dropout(0.2),
            
            # Second LSTM layer
            LSTM(32, return_sequences=False),
            BatchNormalization(),
            Dropout(0.2),
            
            # Dense layers
            Dense(16, activation='relu'),
            Dropout(0.1),
            Dense(1)  # Single output for energy consumption
        ])
        
        # Compile model
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='mse',
            metrics=['mae']
        )
        
        self.model = model
        return model
    
    def train(self, X_train, y_train, X_val, y_val, epochs=100, batch_size=32):
        """Train the LSTM model"""
        if self.model is None:
            self.build_model()
        
        # Callbacks
        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=20,
            restore_best_weights=True
        )
        
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=10,
            min_lr=1e-6
        )
        
        # Train model
        self.history = self.model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stopping, reduce_lr],
            verbose=1
        )
        
        return self.history
    
    def predict(self, X):
        """Make predictions"""
        return self.model.predict(X)
    
    def evaluate(self, X_test, y_test):
        """Evaluate model performance"""
        y_pred = self.predict(X_test)
        
        mse = mean_squared_error(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        rmse = np.sqrt(mse)
        
        metrics = {
            'MSE': mse,
            'RMSE': rmse,
            'MAE': mae,
            'R2': r2
        }
        
        return metrics
    
    def save_model(self, filepath):
        """Save trained model"""
        self.model.save(filepath)
    
    def load_model(self, filepath):
        """Load trained model"""
        self.model = tf.keras.models.load_model(filepath)

### 2.3 `SimpleModels`

This class manages the training and prediction for simpler machine learning models like Linear Regression, Ridge, and Random Forest. These models serve as baselines and components of the final ensemble.

In [7]:
class SimpleModels:
    def __init__(self):
        self.models = {}
        self.scalers = {}
        self.feature_subsets = {}
        
    def prepare_features_for_simple_models(self, X_sequences, feature_subset_names):
        """Convert LSTM sequences to flat features for simple models"""
        # Take the last timestep from sequences (most recent features)
        X_flat = X_sequences[:, -1, :]  # Shape: (samples, features)
        
        # Create different feature subsets
        # Note: This assumes you know the feature order from preprocessing
        # The feature order is: temporal (12), electrical (6), lag (4), building (1)
        feature_maps = {
            'temporal': slice(0, 12),
            'electrical': slice(12, 18),
            'lag_features': slice(18, 22),
            'building': slice(22, 23),
            'all_features': slice(None)   # All features
        }
        
        feature_data = {}
        for subset_name in feature_subset_names:
            if subset_name in feature_maps:
                feature_data[subset_name] = X_flat[:, feature_maps[subset_name]]
        
        return feature_data
    
    def train_models(self, X_train_sequences, y_train, X_val_sequences, y_val):
        """Train simple models on different feature subsets"""
        feature_subset_names = ['temporal', 'electrical', 'lag_features', 'building', 'all_features']
        
        # Prepare flat features
        train_features = self.prepare_features_for_simple_models(X_train_sequences, feature_subset_names)
        val_features = self.prepare_features_for_simple_models(X_val_sequences, feature_subset_names)
        
        # Train models for each subset
        for subset_name, X_subset in train_features.items():
            # Scale features for this subset
            scaler = StandardScaler()
            X_subset_scaled = scaler.fit_transform(X_subset)
            X_val_subset_scaled = scaler.transform(val_features[subset_name])
            
            self.scalers[subset_name] = scaler
            
            # Linear Regression
            lr_model = LinearRegression()
            lr_model.fit(X_subset_scaled, y_train)
            self.models[f'lr_{subset_name}'] = lr_model
            
            # Ridge Regression (regularized)
            ridge_model = Ridge(alpha=1.0)
            ridge_model.fit(X_subset_scaled, y_train)
            self.models[f'ridge_{subset_name}'] = ridge_model
            
            # Random Forest
            rf_model = RandomForestRegressor(
                n_estimators=100,
                max_depth=10,
                random_state=42,
                n_jobs=-1
            )
            rf_model.fit(X_subset_scaled, y_train)
            self.models[f'rf_{subset_name}'] = rf_model
            
        # Evaluate all models
        self.evaluate_all_models(X_val_sequences, y_val)
        
    def predict_single_model(self, model_name, X_sequences):
        """Predict using a single simple model"""
        # Extract subset name from model name
        subset_name = '_'.join(model_name.split('_')[1:])
        
        # Prepare features
        feature_data = self.prepare_features_for_simple_models(
            X_sequences, [subset_name]
        )
        X_subset = feature_data[subset_name]
        
        # Scale features
        X_subset_scaled = self.scalers[subset_name].transform(X_subset)
        
        # Predict
        return self.models[model_name].predict(X_subset_scaled)
    
    def evaluate_all_models(self, X_val, y_val):
        """Evaluate all simple models"""
        results = {}
        
        for model_name in self.models.keys():
            try:
                y_pred = self.predict_single_model(model_name, X_val)
                
                mse = mean_squared_error(y_val, y_pred)
                mae = mean_absolute_error(y_val, y_pred)
                r2 = r2_score(y_val, y_pred)
                
                results[model_name] = {
                    'MSE': mse,
                    'RMSE': np.sqrt(mse),
                    'MAE': mae,
                    'R2': r2
                }
            except Exception as e:
                print(f"Error evaluating {model_name}: {e}")
                
        return results

### 2.4 `EnergyEnsemble`

This class orchestrates the entire ensemble modeling process. It trains the LSTM and simple models, calculates optimal weights based on validation performance, and makes final predictions by combining the outputs of the best models.

In [8]:
class EnergyEnsemble:
    def __init__(self, input_shape):
        self.lstm_model = DecodeLSTM(input_shape)
        self.simple_models = SimpleModels()
        self.weights = {}
        self.validation_scores = {}
        
    def fit(self, X_train, y_train, X_val, y_val, epochs=100, batch_size=32):
        """Train all models in the ensemble"""
        print("Training LSTM model...")
        self.lstm_model.train(X_train, y_train, X_val, y_val, epochs, batch_size)
        
        print("Training simple models...")
        self.simple_models.train_models(X_train, y_train, X_val, y_val)
        
        print("Calculating LSTM-focused ensemble weights...")
        self.calculate_lstm_focused_weights(X_val, y_val)
        
    def calculate_lstm_focused_weights(self, X_val, y_val):
        """LSTM-focused ensemble weight calculation"""
        # Get predictions from all models
        lstm_pred = self.lstm_model.predict(X_val).flatten()
        lstm_r2 = r2_score(y_val, lstm_pred)
        
        print(f"LSTM R² on validation: {lstm_r2:.4f}")
        
        # Get simple model predictions and scores
        model_scores = {'lstm': lstm_r2}
        
        for model_name in self.simple_models.models.keys():
            try:
                pred = self.simple_models.predict_single_model(model_name, X_val)
                r2 = r2_score(y_val, pred)
                model_scores[model_name] = r2
                print(f"{model_name} R² on validation: {r2:.4f}")
            except:
                continue
        
        # **STRATEGY 1**: If LSTM is clearly best, make it dominant
        best_r2 = max(model_scores.values())
        second_best_r2 = sorted(model_scores.values(), reverse=True)[1] if len(model_scores) > 1 else 0
        
        if lstm_r2 >= best_r2 and lstm_r2 > 0.35:  # LSTM is best and good
            print(f"🎯 LSTM-Dominant Strategy: LSTM is best performer (R² = {lstm_r2:.4f})")
            
            # Give LSTM 70-80% weight
            lstm_weight = 0.75
            remaining_weight = 1.0 - lstm_weight
            
            # Find best simple model for complement
            simple_models = {k: v for k, v in model_scores.items() if k != 'lstm' and v > 0.25}
            
            if simple_models:
                # Give remaining weight to best simple model(s)
                best_simple = max(simple_models.items(), key=lambda x: x[1])
                
                if len(simple_models) == 1:
                    # Only one good simple model
                    self.weights = {
                        'lstm': lstm_weight,
                        best_simple[0]: remaining_weight
                    }
                else:
                    # Multiple good simple models - distribute remaining weight
                    good_simple = {k: v for k, v in simple_models.items() if v >= best_simple[1] * 0.8}
                    simple_total = sum(good_simple.values())
                    
                    self.weights = {'lstm': lstm_weight}
                    for model, score in good_simple.items():
                        self.weights[model] = remaining_weight * (score / simple_total)
            else:
                # No good simple models - use LSTM only
                self.weights = {'lstm': 1.0}
                print("No good simple models found. Using LSTM only.")
        
        # **STRATEGY 2**: Multiple competitive models
        elif len([r for r in model_scores.values() if r > 0.3]) >= 2:
            print("🔗 Multi-Model Strategy: Multiple competitive models found")
            
            # Only use models with R² > 0.3
            good_models = {k: v for k, v in model_scores.items() if v > 0.3}
            
            # Performance-based weights with LSTM bias
            weights = {}
            for model, r2 in good_models.items():
                if model == 'lstm':
                    weights[model] = r2 * 1.5  # 50% bonus for LSTM
                else:
                    weights[model] = r2
            
            # Normalize
            total_weight = sum(weights.values())
            self.weights = {k: w/total_weight for k, w in weights.items()}
        
        # **STRATEGY 3**: Fallback to best model only
        else:
            print("📊 Best-Model-Only Strategy: Using single best performer")
            best_model = max(model_scores.items(), key=lambda x: x[1])
            self.weights = {best_model[0]: 1.0}
        
        # Store validation scores
        self.validation_scores = model_scores
        
        print("LSTM-focused ensemble weights:", {k: f"{v:.4f}" for k, v in self.weights.items()})
        
        # **VALIDATION**: Test ensemble performance
        ensemble_pred = self.predict(X_val)
        ensemble_r2 = r2_score(y_val, ensemble_pred)
        
        print(f"Ensemble validation R²: {ensemble_r2:.4f}")
        print(f"LSTM validation R²: {lstm_r2:.4f}")
        
        # **SAFETY CHECK**: If ensemble is worse than LSTM, use LSTM only
        if ensemble_r2 < lstm_r2 * 0.95:  # Allow 5% tolerance
            print("⚠️  Ensemble underperforming LSTM. Switching to LSTM-only.")
            self.weights = {'lstm': 1.0}
            
            # Re-test
            ensemble_pred = self.predict(X_val)
            ensemble_r2 = r2_score(y_val, ensemble_pred)
            print(f"LSTM-only validation R²: {ensemble_r2:.4f}")
        
    def predict(self, X_test):
        """Make ensemble predictions"""
        predictions = {}
        
        # LSTM prediction
        if 'lstm' in self.weights:
            lstm_pred = self.lstm_model.predict(X_test).flatten()
            predictions['lstm'] = lstm_pred
        
        # Simple model predictions
        for model_name in self.simple_models.models.keys():
            if model_name in self.weights:
                try:
                    pred = self.simple_models.predict_single_model(model_name, X_test)
                    predictions[model_name] = pred
                except:
                    continue
        
        # Weighted ensemble
        if len(predictions) == 0:
            return self.lstm_model.predict(X_test).flatten()
        
        ensemble_pred = np.zeros_like(list(predictions.values())[0])
        for model_name, weight in self.weights.items():
            if model_name in predictions:
                ensemble_pred += weight * predictions[model_name]
        
        return ensemble_pred
    
    def evaluate(self, X_test, y_test):
        """Evaluate ensemble performance"""
        y_pred = self.predict(X_test)
        
        return {
            'MSE': mean_squared_error(y_test, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
            'MAE': mean_absolute_error(y_test, y_pred),
            'R2': r2_score(y_test, y_pred)
        }
    
    def compare_models(self, X_test, y_test):
        """Compare ensemble with individual models"""
        results = {}
        
        # Ensemble performance
        results['ensemble'] = self.evaluate(X_test, y_test)
        
        # LSTM performance
        lstm_pred = self.lstm_model.predict(X_test).flatten()
        results['lstm_only'] = {
            'MSE': mean_squared_error(y_test, lstm_pred),
            'RMSE': np.sqrt(mean_squared_error(y_test, lstm_pred)),
            'MAE': mean_absolute_error(y_test, lstm_pred),
            'R2': r2_score(y_test, lstm_pred)
        }
        
        # Only include decent simple models
        for model_name in self.simple_models.models.keys():
            try:
                pred = self.simple_models.predict_single_model(model_name, X_test)
                r2 = r2_score(y_test, pred)
                if r2 > 0.25:  # Only decent models
                    results[model_name] = {
                        'MSE': mean_squared_error(y_test, pred),
                        'RMSE': np.sqrt(mean_squared_error(y_test, pred)),
                        'MAE': mean_absolute_error(y_test, pred),
                        'R2': r2
                    }
            except:
                continue
                
        return results

## 3. Main Execution Block

This section contains the main logic from `main_campus.py`. It sets configurations, runs the preprocessing and training pipeline, and finally evaluates the models, printing a detailed analysis.

In [ ]:
def main():
    # Configuration
    # Assuming the notebook is in the project root, and the dataset is in 'energy_dataset'
    project_root = os.getcwd() 
    DATASET_PATH = os.path.join(project_root, "energy_dataset")
    
    # Check if dataset exists
    if not os.path.exists(DATASET_PATH):
        print(f"Dataset not found at: {DATASET_PATH}")
        print("Please ensure the 'energy_dataset' directory is in the same folder as this notebook.")
        return
    
    WINDOW_SIZE = 10  # Reduced window size
    TARGET_COLUMN = 'energy_consumption'
    SAMPLE_SIZE = 50000  # Manageable sample size
    
    print("="*60)
    print("CAMPUS IoT ENERGY MANAGEMENT - ENSEMBLE ANALYSIS")
    print("="*60)
    
    try:
        # Initialize preprocessor
        preprocessor = CampusEnergyPreprocessor(DATASET_PATH)
        
        # Prepare complete dataset
        print("\n1. DATA PREPARATION")
        print("-" * 30)
        data = preprocessor.prepare_complete_dataset(TARGET_COLUMN, sample_size=SAMPLE_SIZE)
        
        # Display dataset statistics
        print(f"\nDataset Overview:")
        print(f"- Total samples: {len(data):,}")
        print(f"- Date range: {data['timestamp'].min()} to {data['timestamp'].max()}")
        print(f"- Buildings: {list(data['building_type'].unique())}")
        print(f"- Energy consumption range: {data['energy_consumption'].min():.2f} to {data['energy_consumption'].max():.2f}")
        
        # Building-wise statistics
        print(f"\nBuilding-wise Statistics:")
        building_stats = data.groupby('building_type')['energy_consumption'].agg(['count', 'mean', 'std']).round(2)
        print(building_stats)
        
        # Get feature subsets
        feature_subsets = preprocessor.create_feature_subsets()
        all_feature_cols = feature_subsets['all_features']
        
        print(f"\nFeature Groups:")
        for group, features in feature_subsets.items():
            print(f"- {group}: {len(features)} features")
        
        # Create sequences
        print(f"\n2. SEQUENCE CREATION")
        print("-" * 30)
        X, y = preprocessor.create_sequences(data, all_feature_cols, TARGET_COLUMN, WINDOW_SIZE)
        
        if len(X) == 0:
            print("Error: No sequences created!")
            return
            
        print(f"Sequence data shape: X={X.shape}, y={y.shape}")
        
        # Split data
        X_train, X_val, X_test, y_train, y_val, y_test = preprocessor.split_data(X, y)
        
        if len(X_train) == 0:
            print("Error: No training data available!")
            return
        
        # Initialize and train ensemble
        print(f"\n3. MODEL TRAINING")
        print("-" * 30)
        input_shape = (X_train.shape[1], X_train.shape[2])
        ensemble = EnergyEnsemble(input_shape)
        
        print("Training ensemble model...")
        ensemble.fit(X_train, y_train, X_val, y_val, epochs=20, batch_size=32)  # Reduced epochs for testing
        
        # Evaluate models
        print(f"\n4. MODEL EVALUATION")
        print("-" * 30)
        results = ensemble.compare_models(X_test, y_test)
        
        # Print detailed results
        print("\n" + "="*60)
        print("MODEL COMPARISON RESULTS")
        print("="*60)
        
        for model_name, metrics in results.items():
            print(f"\n{model_name.upper().replace('_', ' ')}:")
            print(f"  R² Score: {metrics['R2']:.4f}")
            print(f"  RMSE: {metrics['RMSE']:.2f}")
            print(f"  MAE: {metrics['MAE']:.2f}")
        
        # **DETAILED INDIVIDUAL MODEL ANALYSIS - ADDED HERE**
        print("\n" + "="*60)
        print("DETAILED INDIVIDUAL MODEL ANALYSIS")
        print("="*60)

        # Test all models individually
        print("\nAll Model Performance on Test Set:")
        print("-" * 40)

        # LSTM
        lstm_pred = ensemble.lstm_model.predict(X_test).flatten()
        lstm_r2 = r2_score(y_test, lstm_pred)
        lstm_rmse = np.sqrt(mean_squared_error(y_test, lstm_pred))
        lstm_mae = mean_absolute_error(y_test, lstm_pred)
        print(f"LSTM: R² = {lstm_r2:.4f}, RMSE = {lstm_rmse:.2f}, MAE = {lstm_mae:.2f}")

        # All simple models
        print("\nSimple Models Performance:")
        simple_model_results = {}
        for model_name in ensemble.simple_models.models.keys():
            try:
                pred = ensemble.simple_models.predict_single_model(model_name, X_test)
                r2 = r2_score(y_test, pred)
                rmse = np.sqrt(mean_squared_error(y_test, pred))
                mae = mean_absolute_error(y_test, pred)
                simple_model_results[model_name] = {'r2': r2, 'rmse': rmse, 'mae': mae}
                print(f"{model_name}: R² = {r2:.4f}, RMSE = {rmse:.2f}, MAE = {mae:.2f}")
            except Exception as e:
                print(f"{model_name}: Error - {e}")

        # Ensemble performance breakdown
        ensemble_pred = ensemble.predict(X_test)
        ensemble_r2 = r2_score(y_test, ensemble_pred)
        ensemble_rmse = np.sqrt(mean_squared_error(y_test, ensemble_pred))
        ensemble_mae = mean_absolute_error(y_test, ensemble_pred)
        print(f"\nFinal Ensemble: R² = {ensemble_r2:.4f}, RMSE = {ensemble_rmse:.2f}, MAE = {ensemble_mae:.2f}")

        print("\n" + "="*40)
        print("ENSEMBLE WEIGHT ANALYSIS")
        print("="*40)
        total_weight = sum(ensemble.weights.values())
        print(f"Total weight sum: {total_weight:.4f}")
        
        for model, weight in ensemble.weights.items():
            percentage = weight * 100
            print(f"{model}: Weight = {weight:.4f} ({percentage:.1f}%)")
            
            # Show what this weight means in terms of performance contribution
            if model == 'lstm':
                expected_contrib = weight * lstm_r2
                print(f"  → Expected contribution to ensemble R²: {expected_contrib:.4f}")
            elif model in simple_model_results:
                expected_contrib = weight * simple_model_results[model]['r2']
                print(f"  → Expected contribution to ensemble R²: {expected_contrib:.4f}")

        # Performance analysis
        print("\n" + "="*40)
        print("PERFORMANCE ANALYSIS")
        print("="*40)
        
        # Find best individual model
        all_r2_scores = {'lstm': lstm_r2}
        all_r2_scores.update({k: v['r2'] for k, v in simple_model_results.items()})
        
        best_model = max(all_r2_scores.items(), key=lambda x: x[1])
        worst_model = min(all_r2_scores.items(), key=lambda x: x[1])
        
        print(f"Best individual model: {best_model[0]} (R² = {best_model[1]:.4f})")
        print(f"Worst individual model: {worst_model[0]} (R² = {worst_model[1]:.4f})")
        print(f"Ensemble performance: R² = {ensemble_r2:.4f}")
        
        if ensemble_r2 > best_model[1]:
            improvement = ((ensemble_r2 - best_model[1]) / best_model[1]) * 100
            print(f"✅ Ensemble IMPROVED by {improvement:.2f}% over best individual model")
        else:
            degradation = ((best_model[1] - ensemble_r2) / best_model[1]) * 100
            print(f"❌ Ensemble UNDERPERFORMED by {degradation:.2f}% vs best individual model")
            print(f"   Recommendation: Use {best_model[0]} alone or adjust ensemble weights")

        # Model contribution analysis
        print(f"\nModel Contribution Analysis:")
        weighted_sum = sum([ensemble.weights.get('lstm', 0) * lstm_r2] + 
                          [ensemble.weights.get(k, 0) * v['r2'] for k, v in simple_model_results.items()])
        print(f"Theoretical ensemble R² (weighted sum): {weighted_sum:.4f}")
        print(f"Actual ensemble R²: {ensemble_r2:.4f}")
        print(f"Ensemble synergy effect: {(ensemble_r2 - weighted_sum):.4f}")

        print(f"\nEnsemble weights: {ensemble.weights}")
        
        # Save results summary
        # In a notebook, we can just print this, but let's keep the function for completeness
        # save_results_summary(results, ensemble, lstm_r2, simple_model_results, ensemble_r2)
        
        print("\nAnalysis completed successfully!")

    except Exception as e:
        print(f"Error during analysis: {e}")
        import traceback
        traceback.print_exc()


def save_results_summary(results, ensemble, lstm_r2, simple_model_results, ensemble_r2):
    """Save analysis results to file"""
    project_root = os.getcwd()
    results_dir = os.path.join(project_root, "results")
    os.makedirs(results_dir, exist_ok=True)
    
    with open(os.path.join(results_dir, "analysis_summary.txt"), "w") as f:
        f.write("CAMPUS ENERGY ENSEMBLE ANALYSIS RESULTS\n")
        f.write("="*50 + "\n\n")
        
        f.write("INDIVIDUAL MODEL PERFORMANCE:\n")
        f.write(f"LSTM: R² = {lstm_r2:.4f}\n")
        for model, metrics in simple_model_results.items():
            f.write(f"{model}: R² = {metrics['r2']:.4f}\n")
        f.write(f"Ensemble: R² = {ensemble_r2:.4f}\n\n")
        
        f.write("ENSEMBLE WEIGHTS:\n")
        for model, weight in ensemble.weights.items():
            f.write(f"  {model}: {weight:.4f} ({weight*100:.1f}%)\n")
        
        f.write(f"\nFINAL COMPARISON:\n")
        for model_name, metrics in results.items():
            f.write(f"{model_name.upper()}:\n")
            f.write(f"  R² Score: {metrics['R2']:.4f}\n")
            f.write(f"  RMSE: {metrics['RMSE']:.2f}\n")
            f.write(f"  MAE: {metrics['MAE']:.2f}\n\n")
    
    print(f"Results saved to: {os.path.join(results_dir, 'analysis_summary.txt')}")

## 4. Run the Analysis

Finally, we call the `main()` function to execute the entire pipeline.

In [ ]:
# This is the equivalent of `if __name__ == "__main__":` in a .py file.
# Running this cell will start the analysis.
main()